In [1]:
from steer_core.DataManager import DataManager

from steer_opencell_design import (
    SeparatorMaterial, InsulationMaterial, CurrentCollectorMaterial, TapeMaterial, 
    AnodeMaterial, CathodeMaterial, ConductiveAdditive, Binder,
    TablessCurrentCollector,
    AnodeFormulation, CathodeFormulation,
    Cathode, Anode, Separator,
    Laminate,
    WoundJellyRoll, RoundMandrel, FlatWoundJellyRoll, ZFoldStack, PunchedStack,
    Tape,
    CylindricalCannister, CylindricalLidAssembly, CylindricalTerminalConnector, CylindricalEncapsulation,
    CylindricalCell, PrismaticCell, PouchCell,
    PrismaticContainerMaterial,
    Electrolyte,
    __version__
)

import pandas as pd
from datetime import datetime as dt

In [2]:
# set some standard materials

conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation = InsulationMaterial.from_database("Aluminium Oxide, 95%")
current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
separator_material = SeparatorMaterial.from_database('Polyethylene')
tape_material = TapeMaterial.from_database("Kapton")
prismatic_material = PrismaticContainerMaterial.from_database("Aluminum")

In [3]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=130,
    length=3200,
    coated_width=125,
    insulation_width=2.5,
    thickness=13.5
)

cathode_active_material = CathodeMaterial.from_database("NFM111 (Vendor C)")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=2.93,
    mass_loading=11.27,
    insulation_material=insulation,
    insulation_thickness=3
)

my_cathode.properties

{'Cost': '$ 1.17',
 'Mass': '103.7 g',
 'Coating mass': '88.36 g',
 'Total thickness': '90.4 um',
 'Coating thickness': '38.46 um'}

In [4]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=133,
    length=3250,
    coated_width=128,
    insulation_width=2.5,
    thickness=13.5,
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=8,
    insulation_material=insulation,
    insulation_thickness=3
)

my_anode.properties

{'Cost': '$ 0.68',
 'Mass': '81.19 g',
 'Coating mass': '65.26 g',
 'Total thickness': '159.0 um',
 'Coating thickness': '72.73 um'}

In [5]:
# create the layup 

top_separator = Separator(
    material=separator_material, 
    thickness=12,
    width = 127,
    length = 3600
)

bottom_serparator = Separator(
    material=separator_material, 
    thickness=12,
    width = 127,
    length = 3600,
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_serparator,
    name="CBAK-32140NS"
)

my_layup.get_top_down_view().show()


In [6]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)


tape = Tape(
    material = tape_material,
    thickness=30,
    width=130
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    tape=tape,
    additional_tape_wraps=3,
    collector_tab_crumple_factor=50
)


In [7]:
# make the encapsulation

cathode_terminal_connector = CylindricalTerminalConnector(
    material=prismatic_material,
    radius=16,
    thickness=3,
    fill_factor=0.7
)

anode_terminal_connector = CylindricalTerminalConnector(
    material=prismatic_material,
    radius=16,
    thickness=3,
    fill_factor=0.7
)

lid_assembly = CylindricalLidAssembly(
    material=prismatic_material,
    thickness=6,
    fill_factor=0.7,
)

cannister = CylindricalCannister(
    material=prismatic_material,
    outer_radius=my_jellyroll.radius + 0.3,
    wall_thickness=0.2,
    height = my_jellyroll.total_height + lid_assembly.thickness + cathode_terminal_connector.thickness + anode_terminal_connector.thickness # this will autosize the cannister. Replace this with the measured number from the teardown
)

container = CylindricalEncapsulation(
    cannister=cannister,
    lid_assembly=lid_assembly,
    cathode_terminal_connector=cathode_terminal_connector,
    anode_terminal_connector=anode_terminal_connector
)


In [8]:
# make the electrolyte

my_electrolyte = Electrolyte(
    name="1M NaPF6 in EC:PC:DMC (1:1:1 wt%)",
    density=1.2,
    specific_cost=40,
    color="#FF9D00"
)


In [9]:
# make the cell

my_cell = CylindricalCell(
    reference_electrode_assembly=my_jellyroll,
    encapsulation=container,
    electrolyte=my_electrolyte,
    # operating_voltage_window=(2, 4.2),  # you can leave this out in which case it will default to the maximum allowed by the underlying data
    name="CBAK-32140NS"
)


In [10]:
my_cell.get_cross_section(height=1200, width=1200).show()

In [11]:
my_cell.get_top_down_view(height=1200, width=1200).show()

In [12]:
my_cell.plot_mass_breakdown(width=800, height=800)

In [13]:
my_cell.plot_cost_breakdown(width=800, height=800)

In [14]:
my_cell.get_capacity_plot(width=1300, height=800).show()

In [15]:
print(f"Cost ($): {my_cell.cost}")
print(f"Mass (g): {my_cell.mass}")
print(f"Energy Density (Wh/L): {my_cell.volumetric_energy}")
print(f"Energy (Wh): {my_cell.energy}")
print(f"Energy Density (Wh/kg): {my_cell.specific_energy}")
print(f"Normalized Cost ($/kWh): {my_cell.cost_per_energy}")

Cost ($): 3.93
Mass (g): 260.5
Energy Density (Wh/L): 161.02
Energy (Wh): 22.41
Energy Density (Wh/kg): 86.02
Normalized Cost ($/kWh): 175.45


In [16]:
my_cell.minimum_operating_voltage_range

(0.74, 2.31)

In [17]:
my_cell.maximum_operating_voltage_range

(3.49, 3.92)

In [18]:
my_cell.minimum_operating_voltage = 2

In [19]:
my_cell.get_capacity_plot(width=1300, height=800).show()

In [20]:
print(f"Cost ($): {my_cell.cost}")
print(f"Energy Density (Wh/L): {my_cell.volumetric_energy}")
print(f"Energy (Wh): {my_cell.energy}")
print(f"Energy Density (Wh/kg): {my_cell.specific_energy}")
print(f"Normalized Cost ($/kWh): {my_cell.cost_per_energy}")

Cost ($): 3.93
Energy Density (Wh/L): 139.22
Energy (Wh): 19.37
Energy Density (Wh/kg): 74.37
Normalized Cost ($/kWh): 202.93


In [21]:
db = DataManager()

db.remove_data(table_name='cells', condition="name = 'CBAK-32140NS'")

# determine internal construction strings
if type(my_jellyroll) == WoundJellyRoll:
    internal_construction = 'Wound'
elif type(my_jellyroll) == FlatWoundJellyRoll:
    internal_construction = 'Flat Wound'
elif type(my_jellyroll) == ZFoldStack:
    internal_construction = 'Z-Fold Stack'
elif type(my_jellyroll) == PunchedStack:
    internal_construction = 'Stack'

# determine form factor strings
if type(my_cell) == CylindricalCell:
    form_factor = 'Cylindrical'
elif type(my_cell) == PrismaticCell:
    form_factor = 'Prismatic'
elif type(my_cell) == PouchCell:
    form_factor = 'Pouch'

# insert the cell into the database
db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_cell.name],
    'object': [my_cell.serialize()],
    'form_factor': [form_factor],
    'internal_construction': [internal_construction],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))

db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Cell 2,gASVQwYBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:38:51,0.4.9,Na/Na+
1,Cell 4,gASVBAABAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:39:18,0.4.9,Na/Na+
2,HiNa-NaCR32140-MP10,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:39:48,0.4.9,Na/Na+
3,QNAS-S40160NL,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:40:14,0.4.9,Na/Na+
4,QNAS-S40160RL,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:40:41,0.4.9,Na/Na+
5,UniGrid-NCO32140,gASVxQABAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:41:05,0.4.9,Na/Na+
6,Vendor E NFPP,gASVHwwBAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-12-01 14:41:54,0.4.9,Na/Na+
7,Vendor G NFM,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:42:20,0.4.9,Na/Na+
8,Vendor G NFPP,gASVbwsBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:42:42,0.4.9,Na/Na+
9,Vendor D NFPP,gASVSAwBAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-12-09 11:26:29,0.4.11,Na/Na+
